# MIT-BIH Arrhythmia Dataset

Thesis data preview notebook.

Contents: raw ECG signal, bandpass filtering, R-peak detection,
N/S/V beat windows, class distribution.


In [ ]:
# install required packages
%pip install -q wfdb neurokit2

In [ ]:
# download a few records
import wfdb, os
RECORDS = ['100', '103', '105', '200', '202', '234']
os.makedirs('mitdb', exist_ok=True)
wfdb.dl_database('mitdb', dl_dir='mitdb', records=RECORDS)
print('downloaded:', sorted(os.listdir('mitdb')))

In [ ]:
# patient info from the .hea header comments
# format: '# <age> <sex> ...' then '# <medications>'
import pandas as pd
def patient_info(rec_id):
    lines = [l for l in open(f'mitdb/{rec_id}.hea').read().splitlines() if l.startswith('#')]
    age, sex = lines[0].split()[1], lines[0].split()[2]
    meds = lines[1][2:].strip() if len(lines) > 1 else ''
    return f'{age} {sex}', meds
patients = {r: patient_info(r) for r in RECORDS}
pd.DataFrame([{'record': r, 'patient (age/sex)': patients[r][0],
               'medications': patients[r][1]} for r in RECORDS])

In [ ]:
p100 = patients['100'][0] + ' | meds: ' + patients['100'][1]
# first 10 seconds of record 100 (MLII lead)
import wfdb, numpy as np, matplotlib.pyplot as plt
rec = wfdb.rdrecord('mitdb/100')
fs = rec.fs
signal = rec.p_signal[:, 0]
n = 10 * fs
t = np.arange(n) / fs
plt.figure(figsize=(12, 3))
plt.plot(t, signal[:n], lw=0.7, color='tab:red')
plt.xlabel('time (s)'); plt.ylabel('mV'); plt.grid(alpha=0.3)
plt.title(f'Record 100 ({p100}) — MLII lead @ {fs} Hz (raw)')
plt.show()

In [ ]:
# 0.5-40 Hz bandpass filter
from scipy.signal import butter, sosfiltfilt
sos = butter(4, [0.5, 40], btype='bandpass', fs=fs, output='sos')
filtered = sosfiltfilt(sos, signal)
plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1); plt.plot(t, signal[:n], lw=0.7, color='tab:red')
plt.ylabel('mV (raw)'); plt.grid(alpha=0.3)
plt.title(f'raw vs filtered — record 100 ({p100})')
plt.subplot(2, 1, 2); plt.plot(t, filtered[:n], lw=0.7, color='tab:blue')
plt.xlabel('time (s)'); plt.ylabel('mV (filtered)'); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# R-peak detection
import neurokit2 as nk
_, info = nk.ecg_peaks(filtered, sampling_rate=fs, method='neurokit')
r = info['ECG_R_Peaks']
r_in = r[(r >= 0) & (r < n)]
plt.figure(figsize=(12, 3))
plt.plot(t, filtered[:n], lw=0.7, color='tab:blue')
plt.plot(r_in / fs, filtered[r_in], 'rv', ms=6, label='R peak')
plt.xlabel('time (s)'); plt.ylabel('mV'); plt.legend(); plt.grid(alpha=0.3)
plt.title(f'R-peaks — record 100 ({p100}): {len(r_in)} beats in 10 s (~{len(r_in)*6} bpm)')
plt.show()

In [ ]:
# beat windows per class (0.4 s before / 0.6 s after R)
BEFORE, AFTER = int(0.4*fs), int(0.6*fs)
AAMI = {'N': 'NLR.ej', 'S': 'AaJS', 'V': 'VE'}
fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, (rec_id, group) in zip(axes, [('100','N'), ('100','S'), ('234','V')]):
    r2 = wfdb.rdrecord(f'mitdb/{rec_id}'); ann = wfdb.rdann(f'mitdb/{rec_id}', 'atr')
    sig = sosfiltfilt(sos, r2.p_signal[:, 0]); sig = (sig - sig.mean()) / (sig.std() + 1e-9)
    idx = next(i for i, (s, sym) in enumerate(zip(ann.sample, ann.symbol))
               if sym in AAMI[group] and ann.sample[i] > BEFORE
               and ann.sample[i] + AFTER < len(sig))
    w = sig[ann.sample[idx]-BEFORE : ann.sample[idx]+AFTER]
    tt = (np.arange(len(w)) - BEFORE) / fs
    ax.plot(tt, w, lw=1.2, color='tab:blue'); ax.axvline(0, ls='--', c='gray', lw=0.8)
    ax.set_title(f'class {group} — record {rec_id} ({patients[rec_id][0]})'); ax.set_xlabel('time from R (s)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('z-scored mV')
plt.suptitle('beat windows (360 samples)', y=1.02)
plt.show()

In [ ]:
# class distribution
from collections import Counter
c = Counter()
for rec_id in RECORDS:
    ann = wfdb.rdann(f'mitdb/{rec_id}', 'atr')
    for sym in ann.symbol:
        for g, syms in AAMI.items():
            if sym in syms: c[g] += 1
plt.figure(figsize=(5, 3))
plt.bar(c.keys(), c.values(), color=['#4CAF50', '#FF9800', '#F44336'])
for g, v in c.items(): plt.text(g, v, str(v), ha='center', va='bottom')
plt.title('beat class counts (downloaded records)')
plt.show()
print('S beats are much fewer than N beats')

## Full project

To run the whole pipeline (dataset build, training, quantization):

!git clone https://github.com/md-rejoyan-islam/CardioEdgeAI.git

See docs/RESULTS.md for the steps.
Data: MIT-BIH (PhysioNet) — cite Moody & Mark (2001).
